# Polars, hands-on — *"When the DataFrame Broke"*
### Big Data Analytics · MBA (Business Analytics) · Session 2 · QuickKart

This notebook is the lab that goes with Session 2. It:

- **generates its own sample data** — nothing to download, no file to ship;
- **runs identically on macOS and Windows** (and Linux) — every path uses `pathlib`, every install targets the current interpreter;
- follows the same story as the lecture — filter out tiny orders, group by region, find the top region.

**How to run:** *Kernel → Restart & Run All* (Jupyter) or *Run All* (VS Code). The first cell installs Polars if it is missing.

> Cross-platform notes are called out like this whenever a line was written to behave the same on Mac and Windows.


## 0 · Setup — make sure Polars is installed

We install into **the same Python that is running this notebook** using `sys.executable`.
That is the one line people get wrong: a bare `pip install` can target a *different* Python on
Windows than the kernel you are using. `sys.executable` avoids that on every OS.


In [ ]:
import importlib, subprocess, sys  # standard-library tools for checking/installing packages

def ensure(module_name, pip_name=None):
    "Import a package; if missing, install it into THIS interpreter (Mac/Win/Linux safe)."
    try:
        return importlib.import_module(module_name)  # try to load the package if it's already installed
    except ImportError:
        print(f"Installing {pip_name or module_name} ...")  # tell the user we're about to install it
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or module_name])  # install it into this interpreter
        return importlib.import_module(module_name)  # now that it's installed, load it

pl = ensure("polars")  # make sure Polars is installed, then load it as "pl"
print("Polars version:", pl.__version__)  # show which version of Polars we're using
print("Running on     :", sys.platform)   # 'darwin' = macOS, 'win32' = Windows, 'linux' = Linux


In [ ]:
import random  # standard-library module for generating random numbers
from pathlib import Path  # standard-library module for building file paths safely

# Cross-platform folder + file paths. Path() joins with the RIGHT separator
# on every OS ('/' on Mac, '\\' on Windows) — never hard-code slashes.
DATA_DIR = Path.cwd() / "quickkart_data"  # folder where we'll store our generated data
DATA_DIR.mkdir(exist_ok=True)  # create that folder if it doesn't already exist
CSV_PATH = DATA_DIR / "quickkart_orders.csv"  # full path to the CSV file we will create

print("Data folder:", DATA_DIR)  # show the folder path
print("CSV file   :", CSV_PATH)  # show the CSV file path


## 1 · Generate the sample data

Instead of shipping a CSV, we **build one** so the notebook is self-contained.
This mimics QuickKart's Festive-Sale order log: 50,000 orders across 5 regions and 5 categories,
with a deliberate mix of tiny orders (below 500) and larger ones.

We use only Python's standard `random` module (no NumPy needed) so there are no extra dependencies.


In [ ]:
random.seed(42)   # fix the "random" starting point so everyone gets the same data on every machine

REGIONS    = ["North", "South", "East", "West", "Central"]              # regions to randomly assign to orders
CATEGORIES = ["Groceries", "Electronics", "Apparel", "Pharmacy", "Home"]  # categories to randomly assign to orders
N = 50_000  # how many fake orders we want to generate

rows = []  # empty list that will hold one entry per order
for order_id in range(1, N + 1):        # repeat once for every order, numbered 1 to 50,000
    region   = random.choice(REGIONS)     # pick one region at random for this order
    category = random.choice(CATEGORIES)  # pick one category at random for this order
    # ~30% tiny orders below 500, the rest larger — so the >500 filter actually does something
    if random.random() < 0.30:  # 30% of the time, make this a small order
        order_value = round(random.uniform(50, 500), 2)    # random small order value, rounded to 2 decimals
    else:  # the other 70% of the time
        order_value = round(random.uniform(500, 8000), 2)  # random larger order value, rounded to 2 decimals
    units = random.randint(1, 5)  # random number of items bought, between 1 and 5
    rows.append((order_id, region, category, order_value, units))  # save this order as one row of data

orders = pl.DataFrame(  # build a Polars table (DataFrame) out of all the rows we generated
    rows,
    schema=["OrderID", "Region", "Category", "OrderValue", "Units"],  # name each column
    orient="row",  # tell Polars our data is a list of rows (not columns)
)

# write_csv accepts a pathlib.Path directly and picks the OS-correct path — cross-platform.
orders.write_csv(CSV_PATH)  # save the table to a CSV file on disk
print(f"Wrote {orders.height:,} rows to {CSV_PATH.name}")  # confirm how many rows were written
orders.head()  # preview the first few rows of the table


## 2 · Eager vs. lazy — the core idea of the session

Polars can read a CSV two ways. The difference is the whole lesson:

| Call | Returns | Behaviour |
|---|---|---|
| `pl.read_csv()`  | **DataFrame** | **eager** — reads the whole file *now* (like pandas) |
| `pl.scan_csv()`  | **LazyFrame** | **lazy** — builds a *plan*; reads nothing until you call `.collect()` |

The tell is the word **scan**. Watch the returned *type*.


In [ ]:
eager = pl.read_csv(CSV_PATH)          # runs immediately — reads the whole file into memory now (eager)
lazy  = pl.scan_csv(CSV_PATH)          # builds a plan only — reads nothing yet (lazy)

print("read_csv  ->", type(eager).__name__)   # DataFrame  (eager)
print("scan_csv  ->", type(lazy).__name__)     # LazyFrame  (lazy)


## 3 · The business question (lazy pipeline)

> **Meera asks:** *"Which region had the highest sales last night, ignoring the tiny orders below 500?"*

We describe the whole pipeline first, then trigger it once with **`.collect()`**.
Read it as English: scan → drop tiny orders → group by region → compute count / total / average → sort by total.


In [ ]:
query = (
    pl.scan_csv(CSV_PATH)                                   # lazy: nothing runs yet, just opens a plan to read the CSV
      .filter(pl.col("OrderValue") > 500)                   # step 1: drop tiny orders (500 or below)
      .group_by("Region")                                   # step 2: one group per region
      .agg(                                                 # step 3: compute these numbers for each group
          pl.len().alias("NumberOfOrders"),                 #   - how many orders in this region
          pl.col("OrderValue").sum().round(2).alias("TotalSales"),        #   - total sales in this region
          pl.col("OrderValue").mean().round(2).alias("AverageOrderValue"),#   - average order value in this region
      )
      .sort("TotalSales", descending=True)                  # step 4: sort regions, highest total sales first
)

result = query.collect()    # <-- THIS line actually runs the whole pipeline and gives us a real table
result   # display the resulting table


In [ ]:
top = result.row(0, named=True)  # grab the first row (the top region) as a named, dictionary-like object
print(f"Top region: {top['Region']}  "  # print a friendly summary sentence built from that row's values
      f"(TotalSales = {top['TotalSales']:,.2f} INR across {top['NumberOfOrders']:,} orders)")


## 4 · Why lazy can be faster — see the optimizer

Because Polars sees the **whole plan** before running it, it rewrites the plan to do less work.
`.explain()` prints the optimized plan. Look for two things the lecture named:

- **`SELECTION`** pushed down to the CSV scan → *predicate pushdown* (filter applied while reading).
- **`PROJECT … columns`** at the scan → *projection pushdown* (only the needed columns are read).

Below, we only ever use `Region` and `OrderValue`, so Polars never reads `Category` or `Units`.


In [ ]:
plan = (
    pl.scan_csv(CSV_PATH)                                    # start the same lazy plan as before
      .filter(pl.col("OrderValue") > 500)                    # keep only orders above 500
      .group_by("Region")                                    # group the remaining orders by region
      .agg(pl.col("OrderValue").sum().alias("TotalSales"))    # add up OrderValue per region
)
print(plan.explain())     # print the optimized query plan (read it bottom-up) — nothing is executed yet


## 5 · The same logic in pandas (optional side-by-side)

If pandas is available, here is the identical answer so students can compare syntax.
This cell is guarded — if pandas is not installed it simply skips, so the notebook never
breaks on a machine without pandas.


In [ ]:
try:
    import pandas as pd                               # try to load pandas, a different (older, eager-only) data library
    pdf = pd.read_csv(CSV_PATH)                       # eager: whole file into memory now
    big = pdf[pdf["OrderValue"] > 500]                # keep only rows where OrderValue is above 500
    summary = (big.groupby("Region")["OrderValue"]    # group the filtered rows by region
                  .agg(["count", "sum", "mean"])        # compute count, total, and average per region
                  .sort_values("sum", ascending=False)   # sort so the highest total is first
                  .round(2))                             # round the numbers to 2 decimal places
    print("pandas result (same question):")   # label the output so it's clear what follows
    print(summary)   # show the pandas result
except ImportError:
    print("pandas not installed — skipping the comparison (Polars result above is enough).")  # skip gracefully if pandas is missing


## 6 · Your turn — fill in the blanks

Rebuild the same answer from scratch. Replace each `____` and run the cell.
The logic is the one you already know: **filter → group → aggregate → sort**.


In [ ]:
# result_practice = (
#     pl.scan_csv(CSV_PATH)
#       .filter( ____ )          # keep OrderValue above 500
#       .group_by( ____ )        # group by Region
#       .agg( ____ )             # total sales per region
#       .sort( ____, descending=True )
# )
# print(result_practice.collect())


<details><summary><b>Show model solution</b></summary>

```python
result_practice = (
    pl.scan_csv(CSV_PATH)
      .filter(pl.col("OrderValue") > 500)
      .group_by("Region")
      .agg(pl.col("OrderValue").sum().alias("TotalSales"))
      .sort("TotalSales", descending=True)
)
print(result_practice.collect())
```
</details>


In [ ]:
# --- model solution (run to check your work) ---
solution = (
    pl.scan_csv(CSV_PATH)                                              # lazily open the CSV again
      .filter(pl.col("OrderValue") > 500)                              # drop tiny orders
      .group_by("Region")                                              # group by region
      .agg(pl.col("OrderValue").sum().round(2).alias("TotalSales"))    # total sales per region
      .sort("TotalSales", descending=True)                             # highest total first
)
solution.collect()   # run the plan and show the result


## 7 · Recap

- `pl.read_csv` is **eager**; `pl.scan_csv` is **lazy** — the word *scan* is the tell.
- A lazy pipeline runs only at the **trigger**: `.collect()` (or `.sink_csv()` to write straight to disk).
- Laziness lets the optimizer do **predicate pushdown** and **projection pushdown** — less data read, faster answer.
- The verbs never changed from pandas: **filter → group → aggregate → sort**. Only the engine did.

**Cross-platform checklist used in this notebook:** `sys.executable` for installs · `pathlib.Path`
for every path · standard-library `random` for data · `write_csv`/`scan_csv` take `Path` objects directly.
Nothing here is Mac-only or Windows-only.


In [ ]:
# Optional: clean up the generated file (safe on every OS via pathlib)
# CSV_PATH.unlink(missing_ok=True)      # delete the CSV
# DATA_DIR.rmdir()                       # remove the folder if now empty
print("Done. Generated data is in:", DATA_DIR)   # final message showing where the generated data lives
